# ARC-AGI 2025 — Task Inspector (Colab)

Browse ARC Prize 2025 training tasks visually.

**Run order:** Cell 1 (setup) → Cell 2 (load data) → Cell 3 (pick 10 random) → Cell 4 (look up specific task)

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import os, sys, subprocess, json, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

IN_COLAB = 'google.colab' in sys.modules

# Clone repo (needed for any shared utilities)
GITHUB_USER = 'rodehyde'
REPO        = 'arc-agi-solver'
if IN_COLAB:
    REPO_DIR = f'/content/{REPO}'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git',
                        REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks'
                   else Path(os.getcwd()))
    os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path('/content/drive/MyDrive/ARC-AGI-2025/arc-prize-2025full')
else:
    # Local: data was already converted to per-task files
    DRIVE_DATA = None

print(f'Working directory: {os.getcwd()}')
print(subprocess.run(['git', 'log', '--oneline', '-2'],
                     capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2: Load 2025 tasks ───────────────────────────────────────────────────
# Loads directly from the bundled JSON files in Google Drive (Colab)
# or from the per-task files generated locally.

if IN_COLAB:
    with open(DRIVE_DATA / 'arc-agi_training_challenges.json') as f:
        _challenges = json.load(f)
    with open(DRIVE_DATA / 'arc-agi_training_solutions.json') as f:
        _solutions = json.load(f)

    # Merge into per-task dicts matching the standard ARC format
    TASKS = {}
    for tid, task in _challenges.items():
        merged = {'task_id': tid, 'train': task['train'], 'test': []}
        for i, tp in enumerate(task['test']):
            merged['test'].append({
                'input':  np.array(tp['input']),
                'output': np.array(_solutions[tid][i])
            })
        for pair in merged['train']:
            pair['input']  = np.array(pair['input'])
            pair['output'] = np.array(pair['output'])
        TASKS[tid] = merged
else:
    from src.loader import load_all_tasks
    TASKS = {t['task_id']: t for t in load_all_tasks('arc2025_training')}

TASK_IDS = sorted(TASKS.keys())
print(f'Loaded {len(TASKS)} ARC 2025 training tasks')
print(f'Train-pair counts: ', end='')
from collections import Counter
print(dict(sorted(Counter(len(t['train']) for t in TASKS.values()).items())))

In [ ]:
# ── Display helper ─────────────────────────────────────────────────────────────
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
              '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
_cmap = ListedColormap(ARC_COLORS)
_norm = BoundaryNorm(boundaries=list(range(11)), ncolors=10)

def _show_grid(ax, grid, title=''):
    g = np.array(grid, dtype=np.uint8)
    ax.imshow(g, cmap=_cmap, norm=_norm, interpolation='nearest')
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xticks(np.arange(-0.5, g.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, g.shape[0], 1), minor=True)
    ax.grid(which='minor', color='#555555', linewidth=0.5)
    ax.tick_params(which='minor', size=0)

def show_task_2025(task_id):
    task  = TASKS[task_id]
    train = task['train']
    tests = task['test']
    n_rows = len(train) + len(tests)
    fig, axes = plt.subplots(n_rows, 2, figsize=(6.4, n_rows * 3.2), squeeze=False)
    for row, pair in enumerate(train):
        inp, out = pair['input'], pair['output']
        _show_grid(axes[row][0], inp,  f'Train {row}  input  {inp.shape[0]}×{inp.shape[1]}')
        _show_grid(axes[row][1], out,  f'Train {row}  output {out.shape[0]}×{out.shape[1]}')
    for i, tp in enumerate(tests):
        row = len(train) + i
        inp, out = tp['input'], tp['output']
        _show_grid(axes[row][0], inp, f'Test {i}  input  {inp.shape[0]}×{inp.shape[1]}')
        ax = axes[row][1]
        _show_grid(ax, out, f'Test {i}  output {out.shape[0]}×{out.shape[1]}')
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor('#FF851B'); sp.set_linewidth(4)
    fig.suptitle(task_id, fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

print('Display helper ready.')

In [ ]:
# ── Cell 3: Pick 10 random tasks and display ──────────────────────────────────
SEED = 42  # change to get a different sample
random.seed(SEED)
sample = random.sample(TASK_IDS, 10)
print('Sampled task IDs:', sample)
for tid in sample:
    show_task_2025(tid)

In [ ]:
# ── Cell 4: Display a specific task by ID ────────────────────────────────────
TASK_ID = '00576224'   # ← replace with any task ID
show_task_2025(TASK_ID)